# Decision-focused portfolio learning: guided tour

This notebook walks through the three pieces that make the project work, each on a
problem small enough to run in about a minute on a laptop CPU:

1. **The differentiable portfolio layer**: solve the constrained mean-variance problem and check its
   gradient.
2. **Why the preliminary notebook's DFL model never trained**: the zero-gradient (vertex) problem, and the fix.
3. **Prediction-focused vs decision-focused training** on a synthetic market where the true expected returns
   are known, so regret can be measured exactly.

The full experiments are run by `scripts/reproduce_all.sh`. The last cell shows their headline tables if they
are present in `results/`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, torch, pandas as pd
from dfl_portfolio import PortfolioLayer, mv_closed_form, build_model, TrainConfig, fit_mse, fit_decision
from dfl_portfolio.synthetic import SyntheticMarket, SyntheticSpec
from dfl_portfolio.synthetic_experiments import _to_split, score
from dfl_portfolio.features import FeatureScaler
torch.set_num_threads(2)
pd.set_option("display.precision", 4)

## 1. The portfolio layer

$w^\star(\hat\mu) = \arg\max_w\ \hat\mu^\top w - \lambda\, w^\top\Sigma w - \eta\lVert w - w_{t-1}\rVert_1$
subject to $\mathbf 1^\top w = 1$ and $0 \le w_i \le u$.

The forward pass is a batched ADMM solver polished to the exact active set; the backward pass differentiates
the KKT conditions of that active set (implicit function theorem). The tests in `tests/test_layers.py` check
both against CVXPY and finite differences; here is the quick version against the closed-form solution that
exists when only the budget constraint binds.

In [2]:
torch.manual_seed(0)
n = 5
A = torch.randn(n, n, dtype=torch.float64)
Sigma = (A @ A.T / n + 0.5 * torch.eye(n, dtype=torch.float64)) * 1e-3
mu = (0.002 + 0.001 * torch.randn(n, dtype=torch.float64)).requires_grad_()
layer = PortfolioLayer(lam=5.0, lower=-10.0, upper=10.0)  # loose box -> only the budget binds
w, info = layer(mu[None], Sigma[None])
w_cf = mv_closed_form(mu[None], Sigma, 5.0)
print("max |w - closed form|      :", (w - w_cf).abs().max().item())
g_layer, = torch.autograd.grad(w[0, 0], mu)
g_cf, = torch.autograd.grad(mv_closed_form(mu[None], Sigma, 5.0)[0, 0], mu)
print("max |dw0/dmu - closed form|:", (g_layer - g_cf).abs().max().item())

max |w - closed form|      : 1.1102230246251565e-16
max |dw0/dmu - closed form|: 2.842170943040401e-14


## 2. The zero-gradient problem in the preliminary notebook

With long-only weights and a 30% cap, the optimum sits at a *vertex* of the feasible set whenever the predicted
returns are large relative to $\lambda\Sigma$. At a vertex every weight is pinned to a bound, so small changes in
$\hat\mu$ do not move $w^\star$ and the gradient is exactly zero. The notebook had $\lambda = 1$, weekly
covariances of order $10^{-3}$ and an un-scaled network whose outputs were of order 0.1, which is far past that
point, so its loss stayed at the same value for 40 epochs.

Below: the share of dates with at most one free weight (zero gradient) as the scale of the predictions grows.

In [3]:
rng = np.random.default_rng(0)
N = 10
B = 500
F = rng.standard_normal((N, 3)) * 0.02
Sig = torch.as_tensor(F @ F.T + np.diag(np.full(N, 0.03 ** 2)))[None].expand(B, N, N).contiguous()
rows = []
for lam in [1.0, 5.0]:
    lay = PortfolioLayer(lam=lam, lower=0.0, upper=0.3)
    for scale in [0.001, 0.003, 0.01, 0.03, 0.1]:
        mu_hat = torch.as_tensor(rng.standard_normal((B, N)) * scale)
        with torch.no_grad():
            _, info = lay(mu_hat, Sig)
        rows.append({"lambda": lam, "prediction sd": scale, "zero-gradient share": info.frac_zero_grad,
                     "mean free weights": info.n_free.double().mean().item()})
pd.DataFrame(rows).pivot(index="prediction sd", columns="lambda", values="zero-gradient share")

lambda,1.0,5.0
prediction sd,,
0.001,0.024,0.000
0.003,0.338,0.004
0.010,0.724,0.162
0.030,0.872,0.582
0.100,0.964,0.810


The fix used throughout the project: scale the network output by the standard deviation of the training
targets (so predictions start at the right order of magnitude), initialise the output layer near zero, warm-start
DFL from the MSE-trained model, use a larger $\lambda$, and train on the mean-variance utility instead of the
mean return alone. `results/megacap10_notebook_replication` runs the notebook's set-up and the fixed one
side by side on the same 10 stocks.

## 3. Prediction-focused vs decision-focused on a synthetic market

The true expected return is a degree-$d$ polynomial of a linear index of the features. With $d=1$ a linear
model is correctly specified; with $d=6$ it is not. We train the same linear model with MSE (PF) and with the
realised mean-variance utility through the layer (DFL), and score both by **normalised regret** on 2,000 fresh
dates (0 = oracle that knows $\mu$, 1 = no better than 1/N).

In [4]:
def run(degree, seed=0, n_train=300):
    spec = SyntheticSpec(n_assets=10, n_features=5, degree=degree, signal_sd=0.005)
    mkt = SyntheticMarket(spec)
    tr, va, te = mkt.sample(n_train, seed=seed), mkt.sample(200, seed=10_000 + seed), mkt.sample(2000, seed=20_000 + seed)
    sc = FeatureScaler().fit(tr["X"])
    lay = PortfolioLayer(lam=10.0, lower=0.0, upper=0.4)
    s = float(np.std(tr["Y"]))
    torch.manual_seed(seed)
    pf = build_model("linear", 5, 10, s)
    fit_mse(pf, _to_split(tr, sc), _to_split(va, sc), TrainConfig(loss="mse", epochs=300, patience=50, seed=seed))
    dfl = build_model("linear", 5, 10, s)
    dfl.load_state_dict(pf.state_dict())   # warm start from the MSE solution
    fit_decision(dfl, _to_split(tr, sc), _to_split(va, sc), lay,
                 TrainConfig(loss="utility", lam=10.0, cost=0.0, epochs=200, patience=40, seed=seed))
    out = []
    for name, m in [("PF (MSE)", pf), ("DFL (utility)", dfl)]:
        with torch.no_grad():
            mu_hat = m(_to_split(te, sc).X).double().numpy()
        r = score(lay, mu_hat, te, 10.0)
        out.append({"degree": degree, "method": name, "norm. regret": r["norm_regret"],
                    "MSE vs true mu (x1e6)": r["mse_mu"] * 1e6, "calibration slope": r["calib_slope"]})
    return out

res = pd.DataFrame(run(1) + run(6))
res.set_index(["degree", "method"])

norm. regret  MSE vs true mu (x1e6)  calibration slope
degree method                                                               
1      PF (MSE)             0.0507                 3.0346             1.2222
       DFL (utility)        0.0649                 6.1317             0.8346
6      PF (MSE)             0.2607                17.1583             1.4707
       DFL (utility)        0.1664                18.4346             0.7218

Typically, at degree 1 PF is as good or better: MSE is the right loss when the model is correct. At degree 6
DFL has lower regret even though its MSE is no better and its forecasts are less calibrated. That is RQ1 and RQ3 in
miniature. `results/synthetic_main` repeats this over 5 degrees, 3 sample sizes and 20 seeds.

## 4. Headline results from the full runs (if present)

In [5]:
from IPython.display import Markdown, display
tab = ROOT / "results" / "tables"
for name in ["synthetic_pooled_dfl_vs_pf", "sector9_main_summary", "sector9_main_tests"]:
    f = tab / f"{name}.md"
    if f.exists():
        display(Markdown(f"**{name}**\n\n" + f.read_text()))
    else:
        print(f"{f} not found: run scripts/reproduce_all.sh first")

**synthetic_pooled_dfl_vs_pf**

|   degree |    pairs |   mean_regret_diff |   ci95 |   share_dfl_wins |   share_higher_mse_lower_regret |   t_test_p |   wilcoxon_p |
|---------:|---------:|-------------------:|-------:|-----------------:|--------------------------------:|-----------:|-------------:|
|        1 | 120.0000 |             0.0020 | 0.0094 |           0.4583 |                          0.0917 |     0.6768 |       0.3021 |
|        2 | 120.0000 |            -0.0097 | 0.0102 |           0.5583 |                          0.1750 |     0.0636 |       0.0853 |
|        4 | 120.0000 |            -0.0277 | 0.0134 |           0.6750 |                          0.3167 |     0.0001 |       0.0000 |
|        6 | 120.0000 |            -0.0340 | 0.0140 |           0.6583 |                          0.2750 |     0.0000 |       0.0000 |
|        8 | 120.0000 |            -0.0330 | 0.0152 |           0.6500 |                          0.2417 |     0.0000 |       0.0001 |

**sector9_main_summary**

|                       |   Sharpe |   Sharpe sd (seeds) |   Gross Sharpe |   Ann. return |   Ann. vol |   Max DD |   CVaR95 (wk) |   Turnover / yr |   Utility (wk) |   MSE (x1e-4) |   Rank IC |   Calib. slope |
|:----------------------|---------:|--------------------:|---------------:|--------------:|-----------:|---------:|--------------:|----------------:|---------------:|--------------:|----------:|---------------:|
| 1/N                   |   0.6511 |                     |         0.6549 |        0.1023 |     0.1730 |  -0.5256 |        0.0584 |          0.6179 |        -0.0007 |               |           |                |
| Min-variance          |   0.6951 |                     |         0.7086 |        0.0924 |     0.1420 |  -0.4225 |        0.0469 |          1.8569 |        -0.0000 |               |           |                |
| MV, historical mean   |   0.6475 |                     |         0.6795 |        0.0914 |     0.1539 |  -0.4637 |        0.0521 |          4.9064 |        -0.0004 |        8.7129 |    0.0310 |        -0.5896 |
| PF linear (MSE)       |   0.5531 |              0.0134 |         0.7018 |        0.0832 |     0.1716 |  -0.4351 |        0.0557 |         25.5413 |        -0.0010 |        8.6496 |    0.0124 |         0.6641 |
| PF MLP (MSE)          |   0.5483 |              0.0056 |         0.6959 |        0.0826 |     0.1721 |  -0.4290 |        0.0560 |         25.4286 |        -0.0010 |        8.6464 |    0.0157 |         0.7176 |
| DFL linear            |   0.5668 |              0.0155 |         0.6608 |        0.0801 |     0.1587 |  -0.4680 |        0.0516 |         14.9096 |        -0.0007 |        8.6941 |   -0.0067 |         0.1947 |
| DFL MLP               |   0.5680 |              0.0077 |         0.6603 |        0.0806 |     0.1592 |  -0.4526 |        0.0521 |         14.7032 |        -0.0007 |        8.6890 |   -0.0015 |         0.2459 |
| Hybrid MLP (beta=100) |   0.5657 |              0.0126 |         0.6714 |        0.0809 |     0.1609 |  -0.4633 |        0.0526 |         17.0290 |        -0.0007 |        8.6508 |   -0.0025 |         0.6263 |

**sector9_main_tests**

| method                | vs              |   sharpe_diff_seedavg |   boot_ci_low |   boot_ci_high |   boot_p |   jkm_p |   seed_sharpe_diff |   seed_sharpe_p |   seed_utility_diff |   seed_utility_p |
|:----------------------|:----------------|----------------------:|--------------:|---------------:|---------:|--------:|-------------------:|----------------:|--------------------:|-----------------:|
| DFL linear            | PF linear (MSE) |                0.0170 |       -0.0620 |         0.0978 |   0.6740 |  0.7167 |             0.0137 |          0.0955 |              0.0003 |           0.0017 |
| DFL MLP               | PF MLP (MSE)    |                0.0232 |       -0.0588 |         0.1150 |   0.6102 |  0.6226 |             0.0197 |          0.0092 |              0.0003 |           0.0000 |
| Hybrid MLP (beta=100) | PF MLP (MSE)    |                0.0198 |       -0.0512 |         0.1015 |   0.5952 |  0.6317 |             0.0173 |          0.0159 |              0.0003 |           0.0000 |